In [9]:
from __future__ import division
import fnmatch
import glob
import os
import numpy as np
import gdal
from osgeo.gdalconst import *
from osgeo.gdalnumeric import *
import subprocess
from os.path import dirname as up
from scipy import ndimage
import sys
from datetime import datetime, timedelta
from skimage import morphology, filters
import matplotlib.pyplot as plt
import geopandas
from multiprocess import Pool
import re
import itertools
import geopandas as gpd
import pandas as pd
import rasterio
from osgeo import ogr

In [3]:
## Download Sentinel-2 data from Sentinel Australasia Regional Access (https://copernicus.nci.org.au/sara.client/#/explore)
## using a previouly generated metalink (.meta4) and aria2c
Tile = 'T56HKH' # Specify a tile of interest: T56HKG / T56HKH / T56HKJ
!/p8/fdlausnz/yshendryk/aria2c /p8/fdlausnz/yshendryk/risk_heatmaping/ancillary/{Tile}_L2A.meta4 -d /p8/fdlausnz/yshendryk/data/{Tile}

08/11 10:47:00 [NOTICE] CUID#837 - Redirecting to http://copernicus-dtn.nci.org.au/public/Sentinel-2/MSI/L2A/2018/2018-12/30S150E-35S155E/S2A_MSIL2A_20181226T000231_N0211_R030_T56HKH_20181226T014925.zip
[DL:68MiB][#169c72 1.0GiB/1.0GiB(100%)][#d0999a 864MiB/864MiB(100%)][#d1838f 63
08/11 10:47:07 [NOTICE] Verification finished successfully. file=/p8/fdlausnz/yshendryk/data/T56HKH/S2B_MSIL2A_20190110T000239_N0211_R030_T56HKH_20190110T014544

08/11 10:47:07 [NOTICE] Download complete: /p8/fdlausnz/yshendryk/data/T56HKH/S2B_MSIL2A_20190110T000239_N0211_R030_T56HKH_20190110T014544
[DL:67MiB][#169c72 1.0GiB/1.0GiB(100%)][#d1838f 691MiB/1.0GiB(62%)][#787f75 272
08/11 10:47:10 [NOTICE] CUID#845 - Redirecting to https://copernicus-dtn.nci.org.au/public/Sentinel-2/MSI/L2A/2018/2018-12/30S150E-35S155E/S2B_MSIL2A_20181221T000239_N0211_R030_T56HKH_20181221T014812.zip

08/11 10:47:10 [NOTICE] CUID#845 - Redirecting to http://copernicus-dtn.nci.org.au/public/Sentinel-2/MSI/L2A/2018/2018-12/30S150E-3

In [4]:
## Unzip Sentinel-2 tiles
os.chdir('/p8/fdlausnz/yshendryk/data/T56HKG/') # T56HKG / T56HKH / T56HKJ
!unzip '*'

.SAFE/GRANULE/L2A_T56HKH_A025195_20200419T000241/MTD_TL.xml  
   creating: S2A_MSIL2A_20200419T000241_N0214_R030_T56HKH_20200419T021652.SAFE/GRANULE/L2A_T56HKH_A025195_20200419T000241/QI_DATA/
 extracting: S2A_MSIL2A_20200419T000241_N0214_R030_T56HKH_20200419T021652.SAFE/GRANULE/L2A_T56HKH_A025195_20200419T000241/QI_DATA/MSK_NODATA_B05.gml
 extracting: S2A_MSIL2A_20200419T000241_N0214_R030_T56HKH_20200419T021652.SAFE/GRANULE/L2A_T56HKH_A025195_20200419T000241/QI_DATA/MSK_TECQUA_B12.gml  
 extracting: S2A_MSIL2A_20200419T000241_N0214_R030_T56HKH_20200419T021652.SAFE/GRANULE/L2A_T56HKH_A025195_20200419T000241/QI_DATA/SENSOR_QUALITY.xml
 extracting: S2A_MSIL2A_20200419T000241_N0214_R030_T56HKH_20200419T021652.SAFE/GRANULE/L2A_T56HKH_A025195_20200419T000241/QI_DATA/MSK_SATURA_B11.gml  
 extracting: S2A_MSIL2A_20200419T000241_N0214_R030_T56HKH_20200419T021652.SAFE/GRANULE/L2A_T56HKH_A025195_20200419T000241/QI_DATA/MSK_DEFECT_B06.gml
 extracting: S2A_MSIL2A_20200419T000241_N0214_R030_T56HKH_

In [10]:
def get_date_from_refl_filename(s):
    """
    This function returns the date of 
    Sentinel-2 image acquisition from the file name  
    """
    s = os.path.basename(s)
    m = re.match(r'^S2[AB]_MSIL2A_(\d{8})T\d{6}_.+$', s)
    if m is not None:
        return m.group(1)
    print("bad:"+s)

In [11]:
## Get a list of all downloaded and unzipped Sentinel-2 tiles
root_dir = '/p8/fdlausnz/yshendryk/data/T56HKJ/' # T56HKG / T56HKH / T56HKJ
safe_files = glob.glob(root_dir + '/*.SAFE/')
safe_files = list(map(os.path.normpath, safe_files)) # apply 'function' to every element
safe_files.sort(key=get_date_from_refl_filename) # sort by date
#safe_files


In [12]:
def s2_preprocessing(s2_tile):
    """
    This function preprocess individual Sentinel-2 tiles 
    using .SAFE folder as an input and returns
    a cloud-masked, 12-band composite (Blue, Green, Red, 
    Red Edge 1, Red Edge 2, Red Edge 3, NIR 1, NIR 2, SWIR 1, 
    SWIR 2, NDVI, NBR) with 20 m resolution in .TIF format
    """
    jp2_files_10 = [os.path.join(dirpath, f)
                    for dirpath, dirnames, files in os.walk(s2_tile)
                    for f in fnmatch.filter(files, '*_10m.jp2')]
    jp2_files_10 = sort(jp2_files_10)

    jp2_files_20 = [os.path.join(dirpath, f)
                    for dirpath, dirnames, files in os.walk(s2_tile)
                    for f in fnmatch.filter(files, '*_20m.jp2')]
    jp2_files_20 = sort(jp2_files_20)

    scl_mask = [s for s in jp2_files_20 if 'SCL' in s][0]

    outVRT = os.path.normpath(s2_tile + '/' + os.path.basename(s2_tile.split('.')[0]) + '.vrt')

    outputRefl = os.path.normpath(s2_tile + '/' + os.path.basename(s2_tile.split('.')[0]) + '_refl.tif')
    if os.path.exists(outputRefl):
        os.remove(outputRefl)

    cloud_file = os.path.normpath(s2_tile + '/' + os.path.basename(s2_tile.split('.SAFE')[0]) + '.cloudy')
    if os.path.exists(cloud_file):
        os.remove(cloud_file)

    subprocess.call(['gdalbuildvrt', '-q', '-resolution', 'lowest', '-overwrite',
                     '-separate', outVRT, '-r', 'bilinear',
                     ''.join([s for s in jp2_files_10 if '_B02' in s]),
                     ''.join([s for s in jp2_files_10 if '_B03' in s]),
                     ''.join([s for s in jp2_files_10 if '_B04' in s]),
                     ''.join([s for s in jp2_files_20 if '_B05' in s]),
                     ''.join([s for s in jp2_files_20 if '_B06' in s]),
                     ''.join([s for s in jp2_files_20 if '_B07' in s]),
                     ''.join([s for s in jp2_files_10 if '_B08' in s]),
                     ''.join([s for s in jp2_files_20 if '_B8A' in s]),
                     ''.join([s for s in jp2_files_20 if '_B11' in s]),
                     ''.join([s for s in jp2_files_20 if '_B12' in s]), scl_mask])

    nodata_orig = 0
    scale_factor = 10000
    nodata_new = -32768

    ds = gdal.Open(outVRT, GA_ReadOnly)
    ## Get MASK band
    Mask = np.array(ds.GetRasterBand(11).ReadAsArray())
    Mask_clouds = np.copy(Mask)

    ## RECLASSIFY:
    Mask_clouds[np.where(Mask_clouds == 0)] = 0 # 0 - NO_DATA
    Mask_clouds[np.where(Mask_clouds == 1)] = 1 # 1 - SATURATED_OR_DEFECTIVE
    Mask_clouds[np.where(Mask_clouds == 2)] = 0 # 2 - DARK_AREA_PIXELS (USUALLY WATER/BURNED)
    Mask_clouds[np.where(Mask_clouds == 3)] = 1 # 3 - CLOUD_SHADOWS
    Mask_clouds[np.where(Mask_clouds == 4)] = 0 # 4 - VEGETATION
    Mask_clouds[np.where(Mask_clouds == 5)] = 0 # 5 - NOT_VEGETATED
    Mask_clouds[np.where(Mask_clouds == 6)] = 0 # 6 - WATER
    Mask_clouds[np.where(Mask_clouds == 7)] = 0 # 7 - UNCLASSIFIED (CLOUD_LOW_PROBABILITY / Urban)
    Mask_clouds[np.where(Mask_clouds == 8)] = 1 # 8 - CLOUD_MEDIUM_PROBABILITY
    Mask_clouds[np.where(Mask_clouds == 9)] = 1 # 9 - CLOUD_HIGH_PROBABILITY
    Mask_clouds[np.where(Mask_clouds == 10)] = 1 # 10 - THIN_CIRRUS
    Mask_clouds[np.where(Mask_clouds == 11)] = 0 # 11 - SNOW
    Mask_clouds = Mask_clouds.astype(bool)

    # Calculate valid/invalid pixels while discarding NO_DATA
    Valid_pixels = (Mask == 2).sum() + (Mask == 4).sum() + (Mask == 5).sum() + (Mask == 6).sum() + (Mask == 7).sum() + (Mask == 11).sum()
    Invalid_pixels = (Mask == 1).sum() + (Mask == 3).sum() + (Mask == 8).sum() + (Mask == 9).sum() + (Mask == 10).sum()

    # check if > 95% of pixels in cloud mask are invalid
    print ('Percent of invalid pixels: ' + str(Invalid_pixels / (Valid_pixels + Invalid_pixels)))
    if Invalid_pixels / (Valid_pixels + Invalid_pixels) > 0.9:
        print('This image is mostly cloudy: ' + s2_tile)
        print ('Finished ' + outputRefl)
        cloud_file = open(cloud_file, 'w')
        cloud_file.close()
        os.remove(outVRT)
    else:
        ## Morphology on cloud mask:
        Mask_clouds = morphology.binary_erosion(Mask_clouds, selem=morphology.disk(3))
        Mask_clouds = morphology.binary_dilation(Mask_clouds, selem=morphology.disk(30))
        Mask_clouds = morphology.binary_erosion(Mask_clouds, selem=morphology.disk(10))
        Mask_clouds = Mask_clouds.astype(bool)

        ## Read individual bands
        Blue_02 = np.array(ds.GetRasterBand(1).ReadAsArray())
        Blue_02[Blue_02 == nodata_orig] = nodata_new
        Blue_02[Mask_clouds] = nodata_new

        Green_03 = np.array(ds.GetRasterBand(2).ReadAsArray())
        Green_03[Green_03 == nodata_orig] = nodata_new
        Green_03[Mask_clouds] = nodata_new

        Red_04 = np.array(ds.GetRasterBand(3).ReadAsArray())
        Red_04[Red_04 == nodata_orig] = nodata_new
        Red_04[Mask_clouds] = nodata_new

        RE_05 = np.array(ds.GetRasterBand(4).ReadAsArray())
        RE_05[RE_05 == nodata_orig] = nodata_new
        RE_05[Mask_clouds] = nodata_new

        RE_06 = np.array(ds.GetRasterBand(5).ReadAsArray())
        RE_06[RE_06 == nodata_orig] = nodata_new
        RE_06[Mask_clouds] = nodata_new

        RE_07 = np.array(ds.GetRasterBand(6).ReadAsArray())
        RE_07[RE_07 == nodata_orig] = nodata_new
        RE_07[Mask_clouds] = nodata_new

        NIR_08 = np.array(ds.GetRasterBand(7).ReadAsArray())
        NIR_08[NIR_08 == nodata_orig] = nodata_new
        NIR_08[Mask_clouds] = nodata_new

        NIR_8A = np.array(ds.GetRasterBand(8).ReadAsArray())
        NIR_8A[NIR_8A == nodata_orig] = nodata_new
        NIR_8A[Mask_clouds] = nodata_new

        SWIR_11 = np.array(ds.GetRasterBand(9).ReadAsArray())
        SWIR_11[SWIR_11 == nodata_orig] = nodata_new
        SWIR_11[Mask_clouds] = nodata_new

        SWIR_12 = np.array(ds.GetRasterBand(10).ReadAsArray())
        SWIR_12[SWIR_12 == nodata_orig] = nodata_new
        SWIR_12[Mask_clouds] = nodata_new

        NDVI = np.round(((NIR_08 - Red_04) / (NIR_08 + Red_04)) * scale_factor)
        NDVI[np.isnan(NDVI)] = nodata_new
        NDVI[np.isinf(NDVI)] = nodata_new
        NDVI[Mask_clouds] = nodata_new

        NBR = np.round(((NIR_08 - SWIR_12) / (NIR_08 + SWIR_12)) * scale_factor)
        NBR[np.isnan(NBR)] = nodata_new
        NBR[np.isinf(NBR)] = nodata_new
        NBR[Mask_clouds] = nodata_new


        del Mask_clouds

        stackRast = [Blue_02.astype(np.int16), Green_03.astype(np.int16),
                    Red_04.astype(np.int16), RE_05.astype(np.int16),
                    RE_06.astype(np.int16), RE_07.astype(np.int16),
                    NIR_08.astype(np.int16), NIR_8A.astype(np.int16),
                    SWIR_11.astype(np.int16), SWIR_12.astype(np.int16), 
                    NDVI.astype(np.int16), NBR.astype(np.int16)]

        del Blue_02, Green_03, Red_04, RE_05, RE_06, RE_07, NIR_08, NIR_8A, SWIR_11, SWIR_12, NDVI, NBR

        # GDAL:
        print("Writing image...")
        driver = gdal.GetDriverByName("GTiff")
        dsOut = driver.Create(outputRefl, ds.RasterXSize, ds.RasterYSize, 12, gdal.GDT_Int16)
        CopyDatasetInfo(ds, dsOut)
        # Write raster DATA sets
        for j in range(12):
            outBand = dsOut.GetRasterBand(j + 1)
            outBand.WriteArray(stackRast[j])
            (newmin, newmax) = outBand.ComputeRasterMinMax(0)
            (newmean, newstdv) = outBand.ComputeBandStats(1)
            outBand.SetStatistics(newmin, newmax, newmean, newstdv)
            outBand.FlushCache()
        del dsOut
        # Update statistics:
        subprocess.call('python /d/home/fdlausnz/yshendryk/.conda/envs/fire/bin/gdal_edit.py -stats ' + outputRefl, shell=True)

        print('Finished ' + outputRefl)

In [13]:
## Pre-Process Sentinel-2 tiles using multiprocessing:
if __name__ == '__main__':
    p = Pool(20)
    p.map(s2_preprocessing, safe_files)
    p.close()
    p.join()

encountered in true_divide
Writing image...
Finished /p8/fdlausnz/yshendryk/data/T56HKJ/S2A_MSIL2A_20200120T000231_N0213_R030_T56HKJ_20200120T020404.SAFE/S2A_MSIL2A_20200120T000231_N0213_R030_T56HKJ_20200120T020404_refl.tif
/d/home/fdlausnz/yshendryk/.conda/envs/fire/lib/python3.7/site-packages/ipykernel_launcher.py:121: RuntimeWarning: invalid value encountered in true_divide
Percent of invalid pixels: 0.5757502463495476
/d/home/fdlausnz/yshendryk/.conda/envs/fire/lib/python3.7/site-packages/ipykernel_launcher.py:126: RuntimeWarning: invalid value encountered in true_divide
/d/home/fdlausnz/yshendryk/.conda/envs/fire/lib/python3.7/site-packages/ipykernel_launcher.py:121: RuntimeWarning: invalid value encountered in true_divide
Percent of invalid pixels: 0.7845584453933464
/d/home/fdlausnz/yshendryk/.conda/envs/fire/lib/python3.7/site-packages/ipykernel_launcher.py:126: RuntimeWarning: invalid value encountered in true_divide
Writing image...
Percent of invalid pixels: 0.98766603295941

In [4]:
def s2_mosaic(data, id, start, tile, nodata_new = -32768):
    """
    This function generates a cloud-free Sentinel-2 mosaic 
    using 4 weeks of pre-fire cloud-masked Sentinel-2 imagery:
    data: a list of cloud-masked Sentinel-2 tiles
    id: a unique fire event identifier
    start: a start date of a fire event
    tile: a Sentinel-2 tile name
    nodata_new: a nodata values
    The function returns a cloud-free, 10-band mosaic (Blue, Green, Red, 
    Red Edge 1, Red Edge 2, Red Edge 3, NIR 1, NIR 2, SWIR 1, 
    SWIR 2) with 20 m resolution in .TIF format
    """
    for idx, i in enumerate(S2BetweenDates):
        print(idx)
        print(i)
        datatype = np.int16
        ds = gdal.Open(i, gdal.GA_ReadOnly)

        if idx == 0:
            s2_image_blue = ds.GetRasterBand(1).ReadAsArray().astype(datatype)
            s2_image_green = ds.GetRasterBand(2).ReadAsArray().astype(datatype)
            s2_image_red = ds.GetRasterBand(3).ReadAsArray().astype(datatype)
            s2_image_re1 = ds.GetRasterBand(4).ReadAsArray().astype(datatype)
            s2_image_re2 = ds.GetRasterBand(5).ReadAsArray().astype(datatype)
            s2_image_re3 = ds.GetRasterBand(6).ReadAsArray().astype(datatype)
            s2_image_nir1 = ds.GetRasterBand(7).ReadAsArray().astype(datatype)
            s2_image_nir2 = ds.GetRasterBand(8).ReadAsArray().astype(datatype)
            s2_image_swir1 = ds.GetRasterBand(9).ReadAsArray().astype(datatype)
            s2_image_swir2 = ds.GetRasterBand(10).ReadAsArray().astype(datatype)
            s2_image_ndvi = ds.GetRasterBand(band_id).ReadAsArray().astype(datatype)

        else:
            s2_image_blue_next = ds.GetRasterBand(1).ReadAsArray().astype(datatype)
            s2_image_green_next = ds.GetRasterBand(2).ReadAsArray().astype(datatype)
            s2_image_red_next = ds.GetRasterBand(3).ReadAsArray().astype(datatype)
            s2_image_re1_next = ds.GetRasterBand(4).ReadAsArray().astype(datatype)
            s2_image_re2_next = ds.GetRasterBand(5).ReadAsArray().astype(datatype)
            s2_image_re3_next = ds.GetRasterBand(6).ReadAsArray().astype(datatype)
            s2_image_nir1_next = ds.GetRasterBand(7).ReadAsArray().astype(datatype)
            s2_image_nir2_next = ds.GetRasterBand(8).ReadAsArray().astype(datatype)
            s2_image_swir1_next = ds.GetRasterBand(9).ReadAsArray().astype(datatype)
            s2_image_swir2_next = ds.GetRasterBand(10).ReadAsArray().astype(datatype)
            s2_image_ndvi_next = ds.GetRasterBand(band_id).ReadAsArray().astype(datatype)

            s2_image_blue = np.dstack((s2_image_blue, s2_image_blue_next))
            s2_image_green = np.dstack((s2_image_green, s2_image_green_next))
            s2_image_red = np.dstack((s2_image_red, s2_image_red_next))
            s2_image_re1 = np.dstack((s2_image_re1, s2_image_re1_next))
            s2_image_re2 = np.dstack((s2_image_re2, s2_image_re2_next))
            s2_image_re3 = np.dstack((s2_image_re3, s2_image_re3_next))
            s2_image_nir1 = np.dstack((s2_image_nir1, s2_image_nir1_next))
            s2_image_nir2 = np.dstack((s2_image_nir2, s2_image_nir2_next))
            s2_image_swir1 = np.dstack((s2_image_swir1, s2_image_swir1_next))
            s2_image_swir2 = np.dstack((s2_image_swir2, s2_image_swir2_next))
            s2_image_ndvi = np.dstack((s2_image_ndvi, s2_image_ndvi_next))

            print(s2_image_blue.shape)

    del s2_image_blue_next, s2_image_green_next, s2_image_red_next, s2_image_re1_next, s2_image_re2_next, s2_image_re3_next, \
        s2_image_nir1_next, s2_image_nir2_next, s2_image_swir1_next, s2_image_swir2_next, s2_image_ndvi_next

    rows = 5490
    cols = 5490

    # Find indices of pixels with heighest NDVI:
    max_ndvi_idx = np.argmax(s2_image_ndvi, axis=2)

    s2_image_blue_avg = numpy.take_along_axis(s2_image_blue, max_ndvi_idx.reshape((rows,cols,1)), axis=2)
    del s2_image_blue
    s2_image_blue_avg = s2_image_blue_avg.reshape((rows,cols))

    s2_image_green_avg = numpy.take_along_axis(s2_image_green, max_ndvi_idx.reshape((rows,cols,1)), axis=2)
    del s2_image_green
    s2_image_green_avg = s2_image_green_avg.reshape((rows,cols))

    s2_image_red_avg = numpy.take_along_axis(s2_image_red, max_ndvi_idx.reshape((rows,cols,1)), axis=2)
    del s2_image_red
    s2_image_red_avg = s2_image_red_avg.reshape((rows,cols))

    s2_image_re1_avg = numpy.take_along_axis(s2_image_re1, max_ndvi_idx.reshape((rows,cols,1)), axis=2)
    del s2_image_re1
    s2_image_re1_avg = s2_image_re1_avg.reshape((rows,cols))

    s2_image_re2_avg = numpy.take_along_axis(s2_image_re2, max_ndvi_idx.reshape((rows,cols,1)), axis=2)
    del s2_image_re2
    s2_image_re2_avg = s2_image_re2_avg.reshape((rows,cols))

    s2_image_re3_avg = numpy.take_along_axis(s2_image_re3, max_ndvi_idx.reshape((rows,cols,1)), axis=2)
    del s2_image_re3
    s2_image_re3_avg = s2_image_re3_avg.reshape((rows,cols))

    s2_image_nir1_avg = numpy.take_along_axis(s2_image_nir1, max_ndvi_idx.reshape((rows,cols,1)), axis=2)
    del s2_image_nir1
    s2_image_nir1_avg = s2_image_nir1_avg.reshape((rows,cols))

    s2_image_nir2_avg = numpy.take_along_axis(s2_image_nir2, max_ndvi_idx.reshape((rows,cols,1)), axis=2)
    del s2_image_nir2
    s2_image_nir2_avg = s2_image_nir2_avg.reshape((rows,cols))

    s2_image_swir1_avg = numpy.take_along_axis(s2_image_swir1, max_ndvi_idx.reshape((rows,cols,1)), axis=2)
    del s2_image_swir1
    s2_image_swir1_avg = s2_image_swir1_avg.reshape((rows,cols))

    s2_image_swir2_avg = numpy.take_along_axis(s2_image_swir2, max_ndvi_idx.reshape((rows,cols,1)), axis=2)
    del s2_image_swir2
    s2_image_swir2_avg = s2_image_swir2_avg.reshape((rows,cols))

 
    stackRast = [s2_image_blue_avg.astype(np.int16), s2_image_green_avg.astype(np.int16), s2_image_red_avg.astype(np.int16),
                s2_image_re1_avg.astype(np.int16), s2_image_re2_avg.astype(np.int16), s2_image_re3_avg.astype(np.int16),
                s2_image_nir1_avg.astype(np.int16), s2_image_nir2_avg.astype(np.int16), s2_image_swir1_avg.astype(np.int16),
                s2_image_swir2_avg.astype(np.int16)]

    outputMosaic = os.path.normpath('/p8/fdlausnz/yshendryk/data/' + Tile + '_Mosaics2/S2_' + id + '_' + start + '_10bands_ndvi.tif')
    print(outputMosaic)

    if os.path.exists(outputMosaic):
        os.remove(outputMosaic)

    print("Writing image...")
    driver = gdal.GetDriverByName("GTiff")
    dsOut = driver.Create(outputMosaic, ds.RasterXSize, ds.RasterYSize, 10, gdal.GDT_Int16)
    CopyDatasetInfo(ds, dsOut)
    # Write raster DATA sets
    for j in range(10):
        outBand = dsOut.GetRasterBand(j + 1)
        outBand.WriteArray(stackRast[j])
        (newmin, newmax) = outBand.ComputeRasterMinMax(0)
        (newmean, newstdv) = outBand.ComputeBandStats(1)
        outBand.SetStatistics(newmin, newmax, newmean, newstdv)
        outBand.FlushCache()
    del dsOut

    subprocess.call('python /d/home/fdlausnz/yshendryk/.conda/envs/fire/bin/gdal_edit.py -stats ' + outputMosaic, shell=True)

In [5]:
## Generate a list of pre-fire Sentinel-2 imagery for each fire event
## and generate a cloud-free composite
Tile = 'T56HKJ' # T56HKG / T56HKH / T56HKJ
root_dir = '/p8/fdlausnz/yshendryk/data/' + Tile + '/'
mask_dir = '/d/home/fdlausnz/yshendryk/fdlausnz/yshendryk/risk_heatmaping/data/vector/Masks/'
shp_file = mask_dir + 'FireHistory_2018_2020_UTM_' + Tile + '.shp'
fire_data = gpd.read_file(shp_file)
# fire_data.head()

refl_files = [os.path.join(dirpath, f)
                for dirpath, dirnames, files in os.walk(root_dir)
                for f in fnmatch.filter(files, '*_refl.tif')]
refl_files.sort(key=get_date_from_refl_filename) # sort by date

## Pre-Fire (28 days)
for index, row in fire_data.iterrows():
    # Select Sentinel-2 files between two dates ending with fire start date
    print(index)
    print(row['StartDate'])
    fire_date = datetime.strptime(row['StartDate'], '%Y-%m-%d') 
    fire_date_prior = fire_date - timedelta(days=28) #
    S2BetweenDates = []
    while fire_date_prior <= fire_date:
        f = fire_date_prior.strftime('%Y%m%d')
        if any(f in x for x in refl_files):
            S2BetweenDates.append([x for x in refl_files if f in x])
        fire_date_prior += timedelta(1) 
    S2BetweenDates = [item for sublist in S2BetweenDates for item in sublist] # remove empty lists
    print(S2BetweenDates)
    s2_mosaic(data=S2BetweenDates, id=row['FireNo'], start=row['StartDate'], tile=Tile)


0120T014132_refl.tif
(5490, 5490, 5)
5
/p8/fdlausnz/yshendryk/data/T56HKJ/S2A_MSIL2A_20190115T000241_N0211_R030_T56HKJ_20190115T014856.SAFE/S2A_MSIL2A_20190115T000241_N0211_R030_T56HKJ_20190115T014856_refl.tif
(5490, 5490, 6)
6
/p8/fdlausnz/yshendryk/data/T56HKJ/S2B_MSIL2A_20190110T000239_N0211_R030_T56HKJ_20190110T014544.SAFE/S2B_MSIL2A_20190110T000239_N0211_R030_T56HKJ_20190110T014544_refl.tif
(5490, 5490, 7)
/p8/fdlausnz/yshendryk/data/T56HKJ_Mosaics2/S2_19010523470_2019-01-07_10bands_p50.tif
Writing image...
6
2019-01-10
['/p8/fdlausnz/yshendryk/data/T56HKJ/S2A_MSIL2A_20190306T000231_N0211_R030_T56HKJ_20190306T014440.SAFE/S2A_MSIL2A_20190306T000231_N0211_R030_T56HKJ_20190306T014440_refl.tif', '/p8/fdlausnz/yshendryk/data/T56HKJ/S2B_MSIL2A_20190301T000239_N0211_R030_T56HKJ_20190301T014531.SAFE/S2B_MSIL2A_20190301T000239_N0211_R030_T56HKJ_20190301T014531_refl.tif', '/p8/fdlausnz/yshendryk/data/T56HKJ/S2A_MSIL2A_20190224T000241_N0211_R030_T56HKJ_20190224T015314.SAFE/S2A_MSIL2A_2019022

KeyboardInterrupt: 